In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.ct_column import *
from functions.ensembles import *
from scipy.stats import skew, shapiro

In [2]:
dataset = pd.read_excel('../BreastCareCond.xlsx')
data_code = pd.read_excel('../Codificação variaveis.xlsx')

Using the variables she used

In [ ]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Her2 overexpression (with immunohystochemistry) at CB', 'Disease extent', 
                         
                         'Type of 1st intervention', 'Axillary operation', 'SLN procedure']
#this latter ones (last three) are surgery related ones
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']
selected_variables = relevant_variables + date_columns + numerical_variables + categorical_variables

Replace missing values of diagnostic and surgery categorical first

In [4]:
replace_zero = ['N (Regional nodes affected)', 'Type of invasion at CB', 'Grade at CB', 
                'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
replace_one = ['Classification with respect to other lesions']
replace_four = ['Istotype at CB', 'Disease extent']
replace_nine = ['Type of 1st intervention']
replace_missing_values(dataset, replace_zero, 0)
replace_missing_values(dataset, replace_one, 1)
replace_missing_values(dataset, replace_four, 4)
replace_missing_values(dataset, replace_nine, 9)
# replace the missing values with the mode of the column
mode_value = dataset['Side location of the lesion'].mode()[0]
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].fillna(mode_value)

Done
Done
Done
Done


Creating the cT variable like she did

In [5]:
grouped_T(dataset)
cT (dataset)
clinical_staging (dataset)
new_variables = ['cT']
selected_variables = selected_variables + new_variables
categorical_variables = categorical_variables + new_variables
dataset['cT'] = dataset['cT'].replace('TX', np.nan)

In [6]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [7]:
cT = ['cT']
replace_missing_values(dataset, cT, 'unknown')

Done


To reduce the number of values

In [8]:
simplify_2 = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient']
simplify_3 = ['Istotype at CB']
simplify_variable(dataset, simplify_2, 2)
simplify_variable(dataset, simplify_3, 3)

Done
Done


In [125]:
test_normality(dataset, numerical_variables)

,Variable,Skewness,Shapiro p-value,Normality,Has Outliers
0,Age,0.213822,9.449767e-17,Probably not normal,False
1,Ki67 positive cells (%) at CB,0.973698,5.684308e-41,Probably not normal,True


In [9]:
median_imputer(dataset, numerical_variables)

'Done'

To do label encoding

In [10]:
dataset['Number identifying the lesion in the patient'] = dataset['Number identifying the lesion in the patient'].map({'1' : 0, '>1' : 1})
dataset['Side location of the lesion'] = dataset['Side location of the lesion'].map({'L' : 0,'R' : 1})
dataset['Source of referral'] = dataset['Source of referral'].map({1 : 0, 2 : 1})
dataset['N (Regional nodes affected)'] = dataset['N (Regional nodes affected)'].map({'0.0' : 0, '1.0' : 1, '>1' : 2})
dataset['Istotype at CB'] = dataset['Istotype at CB'].map({'1.0' : 1, '2.0' : 2, '>2' : 3})

In [11]:
one_hot_variables = ['N (Regional nodes affected)', 'Classification with respect to other lesions', 'Type of invasion at CB', 'Istotype at CB', 'Grade at CB', 
                     'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 'Her2 overexpression (with immunohystochemistry) at CB', 
                     'Disease extent', 'cT', 'Type of 1st intervention']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables)

In [12]:
ohe_dataset = date_of_diagnosis(ohe_dataset, date_columns)

In [13]:
selected_variables = [col for col in selected_variables if col not in one_hot_variables]
selected_variables = [col for col in selected_variables if col not in date_columns]
selected_variables.extend(new_column_names.tolist())
date_of_diagnosis = ['Date of diagnosis']
selected_variables = selected_variables + date_of_diagnosis

categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())

treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                        'Endocrine therapy performed', 'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs']
selected_variables_treat = selected_variables + treatment_variables

survival_dataset = ohe_dataset[selected_variables_treat].copy()

In [14]:
relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
survival_dataset = remove_zeros_na(survival_dataset, relevant_dates)
survival_dataset = format_date(survival_dataset, relevant_dates)

In [15]:
survival_dataset = survival_days(survival_dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
survival_dataset = tte_negative_patients(survival_dataset)
survival_dataset = filter_EBC(survival_dataset)
survival_dataset.reset_index(drop=True, inplace=True)

Now the treatment variables, just because it was already made

In [16]:
dataset = survival_dataset

In [17]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [18]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [19]:
# Just because this one is different
def simplify_variable_6(dataset, variables, max_value):
    for variable in variables:
        dataset[variable] = dataset[variable].apply(
            lambda x: str(x) if x < max_value else '>6'
        )
    return dataset

In [20]:
one_hot_variables_2 = ['Conventional RT fraction', 'Boost dose administered']
dataset = simplify_variable_6(dataset, one_hot_variables_2, 4)
change_to_categorical(dataset, one_hot_variables_2)

Done


In [21]:
dataset.drop(index=0, inplace=True)
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [39]:
one_hot_variables = ['Endocrine therapy performed','Adjuvant chemotherapy', 'Chemotherapy performed']
def two_to_zero(dataset, variables):
    for variable in variables:
        dataset[variable] = dataset[variable].replace(2, 0)
    return 'Done'

two_to_zero(dataset, one_hot_variables)

bio_variable = 'Adjuvant therapy with biological drugs'
df = encode(dataset, one_hot_variables, 'first')
df_2 = encode(df, one_hot_variables_2, None)
one_hot_dataset = encode_bio_therapy(df_2, bio_variable)

In [40]:
feature_columns = ['Endocrine therapy performed_1', 'Adjuvant chemotherapy_1',
                  'Chemotherapy performed_1', 'Adjuvant therapy with bio drugs', 'Trastuzumab or other drugs', 
                  'Conventional RT fraction_2.0', 'Conventional RT fraction_3.0', 'Conventional RT fraction_>6', 'Conventional RT fraction_0.0', 
                  'Boost dose administered_2.0', 'Boost dose administered_3.0', 'Boost dose administered_>6',	'Boost dose administered_0.0']
feature_columns = feature_columns + categorical_data + numerical_data
bio_variable = ['Adjuvant therapy with biological drugs']
one_hot_variables = one_hot_variables + one_hot_variables_2 + bio_variable
treatment_variables = [col for col in feature_columns if col not in one_hot_variables]

In [41]:
to_remove = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Date of diagnosis']
important_variables = ['PERSON_ID', 'Regional LN recurrence', 'Survival days']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [42]:
survival_dataset = one_hot_dataset[selected_columns]

In [43]:
with open("diagnosis.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")

In [44]:
survival_dataset.to_csv("data_treated_nopr.csv")